# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step tutorial for loading and exploring the FAIR² colorectal cancer survivor dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using `mlcroissant`
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print dataset title and description using metadata attributes
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

List available record sets (`@id`), and for each, list their field and column `@id`s. All references will use `@id` as required.

In [ ]:
# List all record sets by their @id and show schema (fields and their @id)
print("Available record sets:")
record_sets = [rec for rec in getattr(dataset.metadata, 'recordSet', [])]
if not record_sets:
    # Some Croissant schemas use Dataset.recordSet as the list of all record sets
    # If it's not present, try looking at all top-level entities of type RecordSet
    # so we enumerate via dataset.record_sets if available (mlcroissant >=0.7.0)
    try:
        record_sets = [rs['@id'] for rs in dataset.record_sets]
    except AttributeError:
        record_sets = []

if not record_sets:
    # As fallback, try dataset._metadata_json (raw)
    import requests
    meta_json = requests.get(url).json()
    record_sets = []
    for el in meta_json.get('@graph', []):
        if el.get('@type') in ['cr:RecordSet', 'RecordSet', 'http://mlcommons.org/croissant/RecordSet']:
            record_sets.append(el['@id'])

for i, rs_id in enumerate(record_sets):
    print(f"{i+1}. Record Set @id: {rs_id}")
    recset_meta = dataset.get(rs_id)
    print("  Fields:")
    for field in getattr(recset_meta, 'field', []):
        print(f"    - Field @id: {field['@id']}  (name: {field.get('name', '')})")
    if hasattr(recset_meta, 'column'):
        print("  Columns:")
        for col in getattr(recset_meta, 'column', []):
            print(f"    - Column @id: {col['@id']}  (name: {col.get('name','')})")

## 3. Data Extraction

Let's load tables from the record sets into Pandas DataFrames. Use the record set and field `@id`s from the previous cell. This enables downstream analysis and exploratory data analysis.

In [ ]:
# Collect all record set @ids for extraction
record_set_ids = []

# If you obtained record sets dynamically earlier, use the same list here
if 'record_sets' in locals():
    record_set_ids = record_sets

# Otherwise, provide one explicitly
if not record_set_ids:
    record_set_ids = [
        # Example placeholder, update with output from previous cell if needed
        # e.g., 'https://api.app.sen.science/frontiers/7862866/some-recordset-id'
    ]

dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")
        print(f"Sample columns: {df.columns.tolist()[:5]}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For demonstration, pick the first available DataFrame as the main for EDA
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns for record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We now perform basic EDA steps:
- Choose a numeric field and filter rows
- Normalize the field
- Group by a key field, such as cancer type or sex, if available
All field references must use their `@id` identifiers.

In [ ]:
# Select main table to analyze
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Automatically detect likely numeric fields (e.g., Age)
    numeric_candidates = [col for col in df.columns if any(k in col.lower() for k in ['age', 'count', 'n']) and pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Try all integer/float columns
        numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using '{numeric_field_id}' as numeric field (by @id).")
    else:
        print("No obvious numeric field found.")
        numeric_field_id = None

    # Example threshold (change as needed)
    threshold = 50
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a key categorical field: look for sex, diagnosis, or cancer type
        cat_candidates = [
            col for col in df.columns if any(
                k in col.lower() for k in ['sex', 'gender', 'type', 'diagnosis', 'location']
            )
        ]
        if cat_candidates:
            group_field = cat_candidates[0]
            print(f"Grouping by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No categorical group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=df, x=numeric_field_id, bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    # If grouping field exists, show boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(8,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- We loaded and explored the FAIR² colorectal cancer survivors dataset using the `mlcroissant` library and referenced all entities by their `@id` fields.
- We programmatically listed record sets and their schema, loaded all records, and performed basic EDA, including filtering and normalization of numeric features, grouping, and visualization.
- This workflow can be extended to deeper domain-specific analysis or to integrate with other FAIR datasets using Croissant metadata.